In [1]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference, time_inference_torch

import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.exceptions import ConvergenceWarning
import time
import psutil
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# Constants
TRAIN_SIZE = 0.8
RANDOM_STATE = 1082

base_dir = "../../../Datasets/csv/UAVATTACK/"
real_data_path = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"
models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

# Also include any other files you might want to test
other = [
    "syntheticSpoofing_cgan_data.csv",
    "syntheticSpoofing_gan_data.csv"
]

In [3]:
# Load synthetic datasets
synthetic_dfs = {}
# Try all combinations of model/sample size
for model in models_list:
    for n in sample_sizes:
        file_name = f"spoofing_{model}_{n}.csv"
        file_path = os.path.join(base_dir, file_name)
        try:
            synthetic_dfs[file_name] = pd.read_csv(file_path)
            print(f"Loaded {file_name}")
        except Exception as e:
            print(f"Could not load {file_name}: {e}")

# Add other synthetic dataset
for file_name in other:
    file_path = os.path.join(base_dir, file_name)
    try:
        synthetic_dfs[file_name] = pd.read_csv(file_path)
        print(f"Loaded {file_name}")
    except Exception as e:
        print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_csv(real_data_path)
    real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

Loaded spoofing_ctgan_10000.csv
Loaded spoofing_ctgan_100000.csv
Loaded spoofing_ctgan_1000000.csv
Loaded spoofing_gc_10000.csv
Loaded spoofing_gc_100000.csv
Loaded spoofing_gc_1000000.csv
Loaded spoofing_tvae_10000.csv
Loaded spoofing_tvae_100000.csv
Loaded spoofing_tvae_1000000.csv
Loaded syntheticSpoofing_cgan_data.csv
Loaded syntheticSpoofing_gan_data.csv
Loaded real data


In [4]:
# Preprocess real data
if real_df is not None:
    if 'label' not in real_df.columns:
        raise ValueError("Real data does not have 'label' column.")
    X_real = real_df.drop(columns=[col for col in ['label', 'timestamp'] if col in real_df.columns])
    y_real = real_df['label']
    if y_real.dtype == object or y_real.dtype.name == 'category':
        le_real = LabelEncoder()
        y_real = le_real.fit_transform(y_real)
else:
    raise RuntimeError("Real data could not be loaded.")

In [5]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(objective='binary:logistic', random_state=RANDOM_STATE, eval_metric='mae', )
}

In [ ]:
results = []
for synth_name, synth_df in synthetic_dfs.items():
    print(f"Training on synthetic: {synth_name}, testing on real data")

    if 'label' not in synth_df.columns:
        print(f"Skipping {synth_name}: 'label' column not found.")
        continue

    # Drop label and timestamp from synthetic, match columns to real data
    X_synth = synth_df.drop(columns=[col for col in ['label', 'timestamp'] if col in synth_df.columns])
    y_synth = synth_df['label']
    if y_synth.dtype == object or y_synth.dtype.name == 'category':
        le_synth = LabelEncoder()
        y_synth = le_synth.fit_transform(y_synth)

    X_synth_train, _, y_synth_train, _ = train_test_split(
        X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth
    )

    scaler = StandardScaler()
    X_synth_train_scaled = scaler.fit_transform(X_synth_train)
    X_real_scaled = scaler.transform(X_real)

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("Training on synthetic")

        # Track training time
        start_time = time.time()
        model.fit(X_synth_train_scaled, y_synth_train)
        train_time = time.time() - start_time

        print("Evaluating on real data...")
        y_pred = model.predict(X_real_scaled)
        acc = accuracy_score(y_real, y_pred)
        f1 = f1_score(y_real, y_pred, average='weighted', zero_division=0)
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_real, y_pred, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_real, y_pred))

        # Model parameters
        params = model.get_params() if hasattr(model, "get_params") else {}

        # Num epochs (only available for MLPClassifier and certain others)
        if hasattr(model, "n_iter_"):
            num_epochs = model.n_iter_
        else:
            num_epochs = "N/A"

        # Save results
        _row = evaluate(
            y_real,
            y_pred,
            model_name=model_name,
            dataset_name=synth_name,
            family="Synthetic",
            labels=list(range(2)),
            class_names=["benign", "malicious"],
            y_proba=model.predict_proba(X_real_scaled) if hasattr(model, "predict_proba") else None,
            y_train=y_synth_train,
            train_time_sec=train_time,
            num_epochs=num_epochs,
            parameters=str(params),
            seed=RANDOM_STATE,
            n_features=X_real_scaled.shape[1],
            n_train=len(y_synth_train),
            test_set="real_full",
            timing=time_inference(model.predict, X_real_scaled),
        )
        print(summary(_row))
        results.append(_row)


In [ ]:
# Save to CSV
append_row(results, RESULTS_CSV)
print(f"Results appended to {RESULTS_CSV}")